# Native-detail H/L — benchmark 256 ảnh, T4 ×2
Chỉ đo chi phí cache, **không huấn luyện và không đánh giá caption**.
H: cắt bốn vùng ảnh gốc rồi resize224. L: resize cả ảnh224 rồi cắt cùng bốn vùng và phóng224.
Cả hai dùng cùng CLIP frozen và projection frozen từ checkpoint A.
Mẫu khóa: 16 ảnh warmup + 256 ảnh đo từ TRAIN10k; không đọc câu tham chiếu để chọn ảnh.

Thêm ba dataset: **mscoco-2014**, **pair-source-fullframe-ce** (checkpoint A), **visual-cache**.
Chọn **T4 ×2**, Internet ON, **Save & Run All**. Không cần API token.
Sau khi hoàn tất, gửi `native_detail_benchmark_256_review.zip`.
HDF5 nhỏ chỉ là mẫu benchmark; chưa phải full cache để huấn luyện.

In [ ]:
print('NATIVE DETAIL BENCHMARK: START',flush=True)
from pathlib import Path
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/pair-source-fullframe-ce/upt_hidden_caption_A/checkpoint.bin')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
COCO_IMAGES=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/images')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
OUTPUT=Path('/kaggle/working/native_detail_benchmark_256')
for p in (CHECKPOINT,COCO_JSON):assert p.is_file(),p
for p in (COCO_IMAGES,VISUAL_CACHE):assert p.is_dir(),p
assert not OUTPUT.exists(),'Preserve prior output; choose a fresh OUTPUT or session.'


In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, tempfile, zlib
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
# Isolate each bootstrap: embedded tracked overlays from a prior run are intentional edits.
REPO = Path(tempfile.mkdtemp(prefix='Image_Captioning-hidden-native-detail-benchmark-'))/'repo'
LOG = Path('/tmp/upt_hidden_native-detail-benchmark.log')
PROJECT_COMMIT = '879d9a7dd51038a0eb804b83cc996c53fd03d2b0'
def run(args, cwd=None):
    args = list(map(str, args))
    print('\nRunning:', ' '.join(args), flush=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nRunning: ' + ' '.join(args) + '\n'); log.flush()
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f'Command failed ({status}). Saved log: {LOG}')
def checkout(url, path, revision):
    if not path.exists():
        run(['git','clone',url,path])
    assert (path/'.git').is_dir(), path
    assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=path,text=True).strip(), 'Preserve tracked edits; use a fresh runtime path.'
    run(['git','fetch','--depth','1','origin',revision], cwd=path)
    run(['git','checkout','--detach',revision], cwd=path)
checkout('https://github.com/Supzxjee/Image_Captioning.git', REPO, PROJECT_COMMIT)
SOURCES = json.loads(zlib.decompress(base64.b64decode('eNqtfYt22ziS6K+wM+cuyYSmJcWvyMu5x5043d5JJ7mxe/bsKjo8FEXJbEukhqRiuzP+91sPAARBUrYz0zPdFgmgABQK9UIV+P1Fla6ScB5Vkb+5fzG2Xnyl/32MqvRbsj9Pqihd7c2TZRHNk7m1TPJ1UhX3VpTNrVUe38C7MlpvVmm2PLWy3KryIr7eX+fzZGWl601eVKXPEL9m/GxdR+X1Kp2p5z/KPPuaLYp8bW2iCotES+szPKpq2Xa9gX5LK9uI2p8vPsiaF+tomXwFML99enf+Ibx4F9j5JsmidD9epZu9b2m1N4vKZA86iK+HR/bX7Mv53y8uLz59DOzD43g0PDo4PkqSRbI4jGbHh6MkPjx4kxwcDKKDN1FyECWHi9fQ6Lfzs4/BxD84GR4cHhwdef7B4fHJ6PgQfgxORsPB8evp1+zy6h3UGR2dHI3eHB548Gv4ejA6PIFfx4fHx8fDIVT6f7+fvfty9vHqMnCcge/B/32Acuh6DvyFp6HPT7IAXnAZ/cQn92t29uXtrxd/Pw8vfz0L7GS2ODkZLOJ4ECWj16/h9+FxNItPDuLk+M3R7OggSYaLweIoHh4tTgYH82E8ny3enCTHrxez45PFHKb3+ex/Pnw6exd+Pv8SXvx29st5MHo5fHP08nA4ejn6mr3//cMHfn8ZDA8HgwEi/Gs2TxZAA98SB1fP+xattok7/ppZ8A++CXAVqcw9rZL1JsCf/m1aXYfldrFI76jM59+vbH8TFVUarWyXQWAT/7ZIqySskrvKQWrx50ALpUNdeUlWboskjMo4TYP30apMvBJIIrxJ7svgqoAaKQwxq4KRF61W+W2YRRnXc1/ZX79mNkCI8zkQcGBvq8Xeic3j9Itks4pinparTTVZJXEVEtUnTlVEaSanmy6sVZKJdz8FQ0DRwMoLevm9mNhxHudhOren1gLeFlaaWVT3QVYew1OZWH/HiZ0XRV449vndBrpL1G67+nJ28XE4uPElgvJinhQBTjmZc9ceTD1YRevZPLKKsdhufnkdjQ6PHMfOaGvvia09g9lfr6PiZu/bcGy/KqvC0Ufquj6hJ3Hg13VyN0+XSVk5rui8SKptkVnzNK6c26hYbzcBjWcyHh5NvXUS4dqIV8Oj8eh4NNVQuY5uYFWB/5ROihvYi/NsnlZprmNUvQP2UiHKHPtX27M/2G4bW3V71ZwA+8iQfgrsL7/8bHfg+F2CM9yH1t8SYCZQy5olsESJBYMDwlC4LvNtEScBwWyMLQhgUFYCRCU6LJIy/TNxnNHowIN/XY84lP8lkdzS//niw8XH87MvAvRtOofdc52ky+sq4H58BMGlf7HeF1GMPUUri2Fbs/wuKa0NPCXFNxjqdYKsOAE0FetoBTXmwGpXi71Zvs3mUZFCZSS7fD7nMVoIBbizvpAT0fPTh+/BMALnbvCSJ3A/eMlz8O6G8tVQvHJdGsDdAGpBMRTgcipOOK0Jg2RSDJs2KUpH0oIYYrbxozIqiujemTjO3XDvbuC+dO5eAbvcP34FwJ374d49vrsX7+CBIeA/u0ZAhff4qoiyZeIciwHrb6bevLrfJAEMY7HKo+r1SCNoRlu4XOWzaMU0bYz+B6hDAod9B60r2Cy3eXEDmDHYjnz7UzBCphNl987txC6rqNqW9hSo/+2n3z5/OL86t7EYisT2nIc0KKozHJ3QlG9xyqqb9o75Oa+uLcU6ZE1rvS0r2BTIGasEYTHkev9UwNyDdZo5dfchvQyzUrDFRtenSTYP1tGdXh9e9dQWOwnYfOBArT2C7O4PkzcKUVj4n0EXo73IQJ6kc0t0swbqs2Jkumr0OrNj9i8wFwBj9cSbb2lyCy8GByeewi92GpYJcosywAeP24WbpNALEc4+lWv0uoVi0Y0mgD16z51pr1+ecEEFUjEvwk10DzQ6D2f3VbP5y5ao17qMo/g6CfNsdR8CLSaRGOM1sAYe/ksN0j6O+fXRYKAB+McWMLlI44hYo/2BgACnyv9IiIdZOWzx67S0qtt875fPvwNmF0kB1JRYm3STYKenxO2jzDq/OrOW2wi2X5UkSLiz7XyZQNEG4MGK+db5XbzazoG7sdY5z28znPR+Ccu18QiBSAnA8C72P3kscWGLkQobRxsaUIK6BA2XVvuFZ70AjFVht3L8G3a0tyhgQFIh3pc7dx/2NvFQC5AY35S+hRPk31axhbkDXpFo/xYtl6vEVI6jslK/UWM9OnhEbxYPQD6oHunlaa5+oj6zgKmoF3+mxjM1fFQH32ZphXh5llJOrxUiZWEt+z2d23sN9cpTPM9rESwzxngVlaX1ltDryOH5V/Cft4A9ySFJpuB6ArMFtQa6Sqv7EHFW5CsHulzImqS1pncgyZHDA9XN87UPzaPtqgqLbOkcjFw/haEuUTQNPNz5ko17r91aOGyh0ol7Sjs3YPaOiGDBxT0YUunaWyFdwNI4LcUItAvXa78FJcgdQ2c4N5QagAtQYELqJExwGzqauLx2Pe1p5ZJsaaAHdIOQJBjMO1TKTUiMkXbHDlz9mRR5CSLt8Ng7HL7uwIXZCHTE0Yk3BgxOg8noEMwebzA91cugaAxliOVDrQyKxrIdtMHSZploh624ZRPRtUbZ0CbrSmR3IJ4DDeU9a+iZOqf8BzElFuScVmJSkUJHQyBFluBOvYlSA6YvDwwgu9d2oi1n5U6Gw5EH/07bPSj0egqZnkKdpyFqqvWPJpo+iy8oMkunlpnu+AnosbfZTQYs2W5TW5yDYZBmuL3DRU62GUxrjqyeCJCM9ZCMhxbdxUFTRzxtoTtGi2eTeA5YsN7I7aiBShNSZ5b+Y5s4sRfdpWUwcF0PWmh46FiD1Qo0gxLaTBCRw32gVfzP1D3dWfngDdT2D19pDZ7YzZsT2Y9o/lhXe0NssMc97Zl9aahAC9lxnPivA/c/nPg/h2DmARingzlICRfO0/KPPBWLRMsToom9ScjObi1VkYOOMiGtSZiVQcoKEP1kHTutdWyyhN3paRQ0TW2E457OjLerFMzRIgHDrQT7l+q43TPlRY+8WQ8pRBObTViwer3hUX8toddhNeA03Z2Rg6HP6Nd6eviP/jqqn4f2YijBGG7LBFl1vC1Ag6pYVcOFSTNSi8IZqOo1Ly9by3Mr1oYNBc1K8Az7AI0Dz1Ta2W9B/3hNBT0Y1kWu12Br/0p/o/7+Dur+NI5fBJrh1LGohWYI6Yq6PfWGr7vr96rY0OZoMAIeTIPYQYVt9ftgAOM/eR771SYGAnPaphL2GYHMiK/TbyzVUfmKZsA6Qb9OZnl+07Ff84qddmGISmIYumiw5qtvieOiew7IrK4tYAfYShTu21m+zcLkG2pacRKCor7Jy2i1j/rffrOMtPHh4Cb8NiT1EJR2HzQgu+4AjDbRh58Ag65qh4CS6Z0KpuYcfUQwGx4y2VuRyGU1fF+eBtqQ17RoQrn2/zfdvIe/EqCLevKf42WRbzdlQH5MpJzS+ZO6cmwu8bHEdk1tgridwfu4wcQmJAI76uZYshZocciyDpuk+Ry+xd3qjKmPezX6fNClnKA6opd9++PZFSLyHVh5Fx/Cn88/vv31t7MvfwthHcIRGE5+urnPZk1qkCD6yEGbzYekLB1VHzkOUDCKMtDDPMGeDFzEyWrVWB3VnOiBfNAth7ELSMB29tSgTNQ2oQBRQhWMoUoXJxQxhBBVZnLWoEfSHqOJkmbbpN2sre8QkHxbbbYVMqJJkx4uyo95lohayV0Sb0m1j2E3VvbUbXdQgXkbgDmKmxpIwrZ9FPoCAJu46B5uN8RJZzB6dm8niT/L5/cdE6+X00rLNIPVAYbgYEsPuz0ry3SZuTtQILlDdu9oACpq/TFa44YDE7/yQcsI7MtPv395e35p18ox9uSDWFkmVen2jI8PMYjDBxkufebTqQNPkuQ0dAZi44YGzoUu+Z21KdGI3ub0XPGo9GJu5ZVVwWWklwpQfx22SVT/J1nPkvk8mTf4CTKzeYJ+uAI3APsR/NnRwZwc3Y6YkevuAEwTBCR6wpsBU5V9+WmVrNv7ro236pq3OcJBnNj7tnTi0yvyl2uCA8y3KVXeDbhF+jxEj86RduxSD0G3heQiAiaN+hJsHCkqS1Byo2WWg4CNS5abws9ehsR2UA/mvdaSnySOChBytX9UiKRVtIU3hoiXvhn/KsFaUXH/Li2AyefFvTNPi8ckMUmVSjY1lqQW5aoGaNBAy8x94Zc/A8mBr8QZG8u7mZ0Kbyg5rjao7dsGscDsJRg5T7tDGNaOKRj6nGYGytwcWjtp7l9WBWDy4hNI2A5qeoIKxCh1SKPk9QjwCAt+gmWuxk7vZh0UD9jcrip982BTbUphuV3D33shlXcKgUfUDO6r9sh79nsQeufv7AafZgEs65JkT+YtBt2pZ8iJv7JDMIjS5JY0KaF26KIgc/om6P3p4yYho8qcUF6kS7DWV8FTUCT0po5Rt9cTB39OsvzxRX0Uy88YnCdn1GHrEkehjS/lf1jeZ1V09y+qy8hWF2TrItvDgyLHfulv7m13XMvaxU4uhp3gIhFOFq44IQL2Gob4OgxB2oXhGggnDO2x8oniCwfM5FleptV9MBJu7gajani5r25zS7PryY8tD3pq1vbr/gc+MLBgtOKM0rc+5srP3vJwF0ua5dP82nn5BLd0eQ3KzKp+3M7A5IhB9NWv7uvfYDPVneOQk1kU37Rd432e65dYdMZBF0evB9HB7Gh+MIuTxcnB4fBwNpsfHESHI5C2xwcnw+Hr4+PZPDp8HUfRm+PRKD48GRwdvzlcRMeLN4MhcMx3Z1cC2Ggxf/PmzWgEFV6PoN5wFMWL2cFwGB0kx8nRUbQ4nh9gXMdgNjocvpkNgKPF8cGb2fHRYDA7wQgOeVYoTBUKXVAHkMTqDGNHnT7DzqwDNXwMnnHsYsbcAwgtidbjJhXP0LhESgaxUTgcbjDmmmzPnLwcDkYH9B/Xm9l4Xk9D8LcbQGfiEIDm0ZqooNla9ZRAAONmcuLF0kNOMNaPFHFN/XUOezXP0ljOCod5k9x7iYyhgNFOHFuTaR6h3vUcWxifIbNBuShYwraM1AzshvU31V3hIMoqx/4VEIzHS5s0y6hLYF9jG2MyvMVqW15TTIrbsGjEZGFmE6g2dX8K5Ig7zik/a3CtdVqu0Us6tuxX0FRfTVM6IHRzLlNaXvHUYZnqpnC3fYpo01sQDVFXDYRO3UfMJ3kELeKGGJUfOOSF+sWFLAGPJJp4JG4bocR0AsMuPgVVJmhYpI1zc3gDGD98PFYHKj6ImrsCdaAaVlHHxtDNzuCf/9jRXUc/l5tVCnwROPkq2tSdsMusRr9EqCaM6k2RekVOJ+dJtl0nBe5GGssrxIVG0iLeBZdyAi0EVBrntEHBXHFiS9mEJiw1qF8gcnFV21X9osQ5OXZoe0MXbABfPMPsJgPcDgRKoajr8J4CWnBSckf4ur5KpggfmpX7+gCwwJ7uG0Nt4UwzVcnISEvQsFYURxVWuQDsWmT41lUQoCOjJ3APfsyr9xiMw4MWcWWam1za1SEPiwS8UQtG4aSvhu7/QRILgsFYbJVLttKwSxETlCHXgaqevU/BenZ7t3Q6lTh6TVYAHJBCxYVCSeQHSU8oJMU5Oor7b2kJeljIOoEQmX+nd2/xFTei0kB77UyIavXGQGPuKf2C5fjHFkwHWP3Smez2TUnHOqjXHIZD8SiwVLMUaOq+3i+sexxu7vnFOqkiYmaTab1PEPnku6FR4JPuwSE2ixB8YrAU/mgXLDKvQXVcJYZNI/vwow0qVazgEm1S26iqijL4foNS1PnG5yI33jccAIPzqYY0vx8kD+bR8clPa+Hy1TzUsRqqQfBaykd9NYEEshJ6X6OqJzD19sPFZ1gvWGSKfmjw6c8YvwF7Tso8rGuhUjXLs8TKM+stqI6OiMeAUZXpPJFxNhjYmG9cv4M822L9lEI8AmMsdOqIBrqwlhwZBOyhGYTVAhntizjOwhRJBQN8RIBKAtsX1u0Uw0fW9exqJLLuwfEhjLaeIwyxf2kbhqRnJHOOldWs0ZB1r4A0DO1kREpMUSx1EK8ptFXrWg2RsUeiRKgTrX2rn8JoM1KRSSa2OX7K24jV7a9XaYCR3pjQFPLt3wSNWevo3srXIL5k2Wlzd1ppaQmsIdmANQF0ay0SQDOwNBJYAFpYIb7tauohvyPtEKj3RsXGmftc6v0Yp95+42eZvwDjU0RawkZ+r+0LM7blPdkJz982JsvUrYvZNl2JE7BQAfwXOK3ZqIzBvAWjDxfRqoOHFumSj9kKIGxZ7tFmCJmePPESLEswNikISLqBaq2DrXU8imB0IjmDihFW16j+lc5IFsTbeUSlc6CEOHFozQQ76xmNQ6zhlBsEDEa0thGczQt/qpbGpyO2qy+/f3x7dnX+ToauI5hdtsOpEIstp1BjLz3BE0QH360zkwmOcjweTU9FwLQpuVQFIZj+VZYHZhZQCbMkDD7joPhdjNDHcDjH9UG1YQTLlakdaox+xA4vvGZTTb11tMGjRwE03mxt75YigMUAOPoecELkhVwULGxY6Vr9l5GCgdqVHEToHB+deIfDkWtWpKFooBwQo/WYJjb6wEUke4EbJ9SagunEisQNBwTxWG3PnqVRabsPHtOhLppQUtTgW6PWgPegEruD6W9X5Fl3CBNe3cwdc6HUfMoQE3BChzHHIDhEPZjom9RZs9qw7oY6VVYS85Wgk9s4NnJjRLdtqhMLW3IBJNLv+J8HIRFN9mCEkJAJCTvCNRUoDhwih0NbBSZdCsvQFGYfAT/5IljfoZj+Rk8iX2GGVkDDURfdBoQRjk0K6YClDLiev4qAL16n83mSMRk1rAdY6zIAAJOxNxxP0TzA2CJnb+gND/D/QJauv0lgA4MpNfBee0NvpNsfeb4CReC9H31bhvgwmjsCKtTTGo6wqavDP3hDwLVpMBp4Y4AVsnXqBXa4oxrzfMJAVdNMxNiGa0rn0DDzFwuM11Uap9XqnvyfVokyGQQgwkNnBkX5qKSIa0wyyFZmWK+vqxf5TPLoSIT1vDl6ibtX8G/+I2IEuaKMp1fT17H7CDKxv9YCdOG2hsOdGiFUrEVzFxQhN5h6BBwexiP4/9QHTp45AG/oul5R5atgAMwU/7hGCPWT7BtSf5TRIVmDMM9ZJEzGB1PDkuCwUrWnjNDF/onVe5yTE7z6RVfuQv8EWbWAderaUmYv/jYr/7FNkj9hGXRW2LHrwN73QWCYJwclzAnWXNAJyj6KPHakech4aOQtPRknNA2Pe+CJ+oMhT9wfDA7N+ABasIYFJ6PamiMAKXgXRrMyXwH5hQma/AFRuMM97nGPrg91QEBgkoPbON7oMOs6yWM3aZwSfxOYAxTHN86kXp+KxUXciIStj8jbCVrutC3L1MiEnh40OLAxOvaiCJZULtIMrFlHthSxhx0Ono+oFGJdMWnVWcPPo+uW91l8XYBG92cDe7okAxTdt+UYG+Vg86woFcv+fHZ5aXsNMyXgJ48sWjok4kE9ZkkZ3gDWfthmY9ba7CYEEkwKjBEIaFAdtCn0qrCl9emENIeJIrNuDevV8ASTOJU4voaVpuVhfReES4Jqk1B4a49Wa0lNwH+VXYqVvIJyACk92HQmZs3TOXUmupFnS7550EzAy1WSbBx/1L3Y6O+qwk0S3YTrZJ0X98RMMCwYGsOGL4HBnpJPj6Je6FHoGZwaMzB8hUwhksZ0Irk+tB/xAN3u8AA1/DjiHETmS4nk1froslbWmVTaqn0te+WJpe5aMPAo3Atl8D0ei3HEsAeqRAbeObHnYMwlBkyjji2Es70Y2V1s4qFzagZIyQ9L2FjoXdOz97odeWg1aUl2oF4fHZjcvKevu3vRSzM23D0V9cnDZIqWWbJMM2NrhFnr7JpSJxcL6KeOVAYdANB10BW60GFWMiOeTLsjpgRLx9lPuJ8x/3nVEv59nF4jcCmf6LSEcw5RumMUvEb2r9hy71C7jVi57v5pQj5YwNiTJlMA+f1iJWWLp0OydAQjiewSXXixUDkFHaHDD/WkdRBA+U+vMKN12bweSWEm8hmUbnoA+qTcKJ1BfV1iboPa7eMiTrlGO4VcPbrhUYAgfUzzdZ4xiuFR/yjefx4e0ekS6Cq3nf3uUqUANKvxjuvh0B7XpziGGPhBQI1J9/NZuWv6inR5frr84eXv3n1/eHHzOExRrTuWfHMST1s7M6Chg1X4R8e2bnKdUwqwetqoWWg1lczNNuTwPXT7iPiGAN4KFT4pQqZUMomkeyOM8819ANjyrueLw5BjvNjRTr+7Np2mIgGUJYYQ9mhJ4tBRYsOr8ipaYdh+516ms4KFjWEkDGpsfZdNH/YxW1jRvTi87D4wN+UoShGWoPaU6vaye0wm3snon32iw9RvzQDlTt+Y6qBOrjGpZeLUDRBdz8n5kjBA1k0n46lniLq20NpF2k/qCsh+7wB6qkld6yVaQMc/4oMS/In13p8CgtPFjor8zyQTqb3xNYrduaHyA83xqdl27dxNhCvvjtPfaB+5tXNPvAErU9NeSKSESBigGBIUDr5FfLaATTttCbnaPRvlB3NdSDUx01zgj2FIbK6BP9W2ByHE40lRQsvmcKBKmTnDuq+TeRpljjZ3sPb1Rm86GgFZZEhwejPPf+Ma+T1WQ7fWHzxdCdd+M3vi+nwdTDNW35gwKvp4mUsM9Cyz2zVZgZa3sANULRBI1EyE8e5uJStBo4aVRlSqnzqJSMV4sZRKYN95CPzy1zfzFE/0KdaZNXsKJg7zm0D37OoHj8Cn4nwliAn7Oe05tRAiIWHjhuKwSqVtVo3QYDOUynAuAPXWOu6olV5BoANB+Gy06WS/ype2DCC77bpcR4xNSjd+NBiXmoyqpWL6/M8Ee1Lelz7nMWA6k2fvbW1PRmB6YlzA5/b2+DzJ9kQIqYlUF+vg0LkGnSFNPQ5RDnhw+JQURaAN4vLq3affr55uZZ+ipytoGduYu7DxNzkqY3j4iTkaHHmAK6Cw0GFyN1rKm3GwuTdwOyEwb/3CXnrBXeuQTouj4E+tMpGWuAUr2dY8yWsSoPbY4UZ5aPkKNLtOkFPbIUQQSEj2+h5Uqk+9NRo1mmz28ursyxXwWE3Kd8l9t9s3VQ/neW4NRCZ5hPaV98hK7uIkwYvTXg8sEB/bqkOX7/Cf7CG1/HV4OO6xFXdsT7WDgk4VTqxQT5uGivaAQYXOZtdBJ459oxaH8zlswoG1b5zs7wvHne32a3X4D+2T3Uq75hAaNkOjeFPwCUlMVz4NfnwvNBZKOKeCiXkq3JL9D085IG4vY1MtEt2BnqJOSUkCYVydLBu2yjr0Jz1o+xr2jjVPF3QOVAmdShzMNiYL4iEHEdO+cOiUS6Tnqq3WkPKBB/bPc4XmMyGUNTUAFbHbiV2/6brxx8NLWDAh/Ua2olBwn95sS1CtKGvAx2pGn5jOBapWsgnsL5QtUZsfcV5WbAsUyVrcFoOI/McWpidPwViIYhGmA7O3fl8FvWMEfLStcsBfGlvLHMRDTM473/ZM/1Pb3ydO9YVHUERvhOKSGvm+y6vdm9fBC4daAm5x7RI9LlB35HXsTuRhm8r6GZjtOf2EsYyf03eDN4ucF08F3gfql4/uo6gKoUPHfSIOdKpF4hen65jVsbo3ItYFA2jwgrZglRK+KZTH8jUogGu6s8I8k3kCdNTCJJzbKK2cigVIMDKP1gTONYVDypq7TQokqsDcpDjK0wZUY2Csw1CMJGteMkq/+2hpt/anece7Vb82PlGdV/Kb6Y/eYaztK9vCWwjHeA/jq0fy9zw6QwMSEhcz2u5kj2Kxp90SRcTLOazhSo1dfzveES7fnUyF3n1PVb34HL47f/8Bg4s4DY+SSU2qEzGk4oZGtAMKkfdjux2KXTN6mPQQ7YJM9qiKjWUTxuEP5Q9x75zHx64LgdA6UJkyqBqhm5fnH99Zl7+dffhg4anC+X9bMKuxUpbN6TcQvSvtSN7/WQBjD2TKj39WLLcYbvSZ3iPd0g8/muMlBVzmkFADW+c6T2O0ZjDkxNY0+jp5obt5rfGL4Jk5D7ivNuv+dN4AKKn7HXhD6emA6miXUnP6gwDweIl7aiVAYKnPZU8JFKMrDqAFThywKGY6liGNBMejGlqYXLIymkm8jCWJsxlKrTVLVDUvE5loJ0bKCWF1rOC+HivYSA37tKn2MIkXXUog3sRFWiUrzcjOLXHJVpqUfC2wvHQNL12ioMNn3ArMb7TkLxmz3MgSEz+BNVQcMtmV+sVRn5IX9EYaJhnalHPtgkU0DMTbsQjBEcwDFqnc4JQDyx4fDN4cjU8EExRXskBBDqZq9i0tgEqWeFD19vefP5xdhv/96cvfLj+fvT0P3376+P7iFy1jRLb9KdB6IPTWDgtgEujRT/myT6cZ69RWcC+TyurpOBADl7rN29/fnVkKdn0vHulqai6T3nlMLW3YwgNB48bLauYJy1DMoo7DaLXMgWNdr0uK7/TQTMj0MEG9OWoJCWiVOP8s8+ssxMCiyjvqNnqF+nUUaLs6+oKq9Xbl84XB1eL16Ak99FX+i3VmlUD7aO5XFW4L2ASirQXVcmuVrtOqBBrbwyi/ffyPBepHGt/vkfSQOnusrmvtGjOTZ7gAC+o6LOcbZzf+VIN1sg4TkC1xivcHPadhVHFHmuAF0oWdjD5wp6OlZ9miLaEMG4Pg2tWHqrdrUOtrjiVegO2ICAvFTlVt1L2yfGeP8FPzk9w2fK9DIOogvYBG77jiRIpuoVhu8606L2Ag0MDIqETJSf8hgD4dZLueBTYxyHN+RyeIrn61sg5SWldctw4FdH2UwwKtfKWePCjzq1xeVGNcVUoAO/Mpmy57fFC3JfbMzMiqFBirVRsC4msxuK5MVdFZU3OSAKjGwqlR2FwuvhDDxNkTZipHpEER42EZdEkeRCmy7vF20nyD7v4/k0KQA4u2Mo5WUWFdw4oWmD2xTujKSk76SuMbAELLoGSbEZKK2NBR0bwmBEs9QdpXgjTVvcw2j8P2DBomkNPdIHEtTEVTwaW4a8+aTMQYb4BWxc9vrgiLhqXWlhmByoX1LO2u8c3YcpDanQ1m7flSL/QsymPDd647fWSsDu0Tz8LABVebP8GlyWpwJ2qgNM5vIht5bXYCr5RzFbMQW53CAD1MTvQsOurw6EDv0e6pK1PmXkFNFrgL+/es3G42hDUmQmoytr7X4B6My4WNLacZ6nKuRL+4Rtq19yw5zevtu29u126rbkVsK7U9vsFTKWAB3x/qTc/XuKD9QWwHqMYmauWoJHrUMmDwOVtVN/gXj1QbN2yiJay6mSBgVBlUahre5ofhzOLCFsMqVtU+M4BGWmUHWFx3BrLcbBvP6Hpt6FPRN7BGkRs0tClu9t2m7NCx3mSpMliIKpwBrIwNGnQksg3HRMhOd4u6HrST4VSCDr7bm/vqGrA6RgVWIgSAS/0WCuRPX/7A/chLgs2UCA5D0TwMzYM6TpyRUxLVpKgm2WuPu7UduUTYJeAHqmGUAYxPYN+uF8Lss08BVF1FRb+WqGR7aypKcVRwYEF7waja/QBZ96hvtxk/ooVKlDU77WvWqGR2DRqN0mbaAFDfWXbpO60pgOJEapo97lNzSYPTAHAbUiJ7GikF85FudbWyr39T9dy1GDPoN1RWBUB8slVlwuKtFV7TPJJk3gHr8/9c/frp469nl79enuMVPa0ZxvkGx2C/lbZvkcy3YKPRZbeYHlGv75pM4LjIy3KPt/6+jHmdpdUtSg11w7lvP7TMcD1poGGGf4j+vLd+fff+0OIqe1UOyKUDrKSgTkUAWa3FsOMaXRV0h0SVW1/OfjNtcc2cLpJd96/UhjQGtIi7ARyZSN/Ust5+evvJunjHCYmyite4YQYUr79FBfZ0j4GQ+/Wt7NCwqVVRjj8w5SLx0f1Oj05hYyeh83/H5G/4JzT/J8a3uKPB8CB0vn6dv3K/fvX/2CxBKJF3RA3V10RNuhDgpe/XlPHaAcvCfhtl5Bzg228smiZ/X0NO9rvs5afCFPfog+PrCijJ1xnK4AK+11xLINFuMg/J7A9DuorIo5UBJTSd4ydvgq8vRHTR1xemqkkV5Fnxd62iB0uEuFYPN2IV6MWDqTqaOPj6Qn47gclQ3AcE04f+YBjGPahiHIEYkDZGM1dXNaEpQgszTUZZIGZqvnRVQxtaZ9aajEwOUOnxWnPSc4Ruyx5TeS3T9SEe1Wk+WQzfEMecE3w37TxArgF3nM323QaxsD/mFvQornDAFcIeHuyOkC7VAd9NsKM3c0S9QfyPDU9B6DiyFZdlzBGNlVa1vkWjOzZVtRMkWS91z+DqCjIsRIJo39PRC6x9RnpWWciNgf+AfdAg4QVFKyNvZ0dy45SUiRk4152mJ6uCkKPnys6yTTpnryDIGvhtnsFQEAtB9tTxQR2/V8/MXMSu8EVLxi8uusmjDlTEXhYcvqiCMSiwcdER07iTfBr8kcl4LDB6jd+KyOuPxMwtPK8AKWd304jKwVA3Ry7QhLPT9RIDKNXL549HXQqlrhegezkJrsqV6Iy7FvWR5eLyLib1IKcePorBTTtnpGK6YZes0b/7Giek3pK3ZzIcT7HIGb459ixMf/wX5ud89CwFx7OWgLHvze4eepCvapETy78BksRR2Qu7MWQuJY8Aft9ABiONPOvgB8atloM+JjRL2CTHoPTCEhmiPcMFnCukDuVdTvDSxRf4W6WZkRdg3piXGLWdbu3nj1lQjRxxRIIdPxGCyIfdt0qXeC8K7U9FKx2T+Iv1Cb8OQ5/yWoM5jzJUOKdKi5W2U8mhFJ7KCm/VQBdYeeP3Jpdo7rqalcCgJ+PezSwVOhTVWdUpQ3Xsy9qSkxMD23FHbAc+320xDRkFm7zUQuhR3yX0BysiRVpUwHQZ290hLWgQE9kavQGOzl05eVbPUq8vGmLtCn551iqaJY2LsdZpiS79WnHA/CNEkv45AyK9PW0cPt5h1pCGqGoyqMekFBAbjQKj5pGWRTP3QQTLqyGRyqlLsVNgBXTFRAlNRa3JeDiYPtjtDH0xaYGvhgbZpZ11fQaoh3wUzKZSKivpOkAX5TBC/pbc13p3H4XAXolmeAFWGxc6rfzFuuQ7cBJLyuoN7Nh3wPo/4FYrRJSRZ5XRgqMy8VsK+OOG3cGb6FZPek8Xmnj/qSHfu64ob4VfPE0/aFIvUmAHnTcGVTdpYFlqKF1jk2WTui3unVq3qJUQvQprG3oIRhHdyyF2wdRFJ94wN/VBD0PnaOPzeoZqh1mBMpmJepDZTI9uIbDm9kSOlclFWa/uISm7ff8CdaxvH17O1mXQ+GEnmncL7yrJoDtHtEUeXWqlbhTiPeZ00hG2RsHOZ7kOdDgThj7mBjU6oAR8W3ZBp7gN3VWUbyhhU/ejangRd2cYLozWJ9TITKZr/oC9bmCbbipGv2A+Vp3kJL/VpmII6h1qOi66r45FglIXMl3SnTd4Pbw4nJZxA9AF3oUJTGBuxgt0Xv7k6XdAcT98Mlelq7JxpdM7ViY9beB6C3afW/WFtOxCl6P3yalDZ1HyTinG65V8Lys+ch+U13TTyFYysq7RTHRxTje+A87FhVLcokzWmAYSh7xuCrWk5oiXDE62yGciGS1DB77RoFnYbIheEWAUc3HdYp4ZbVvloSAivoi4fr2OyhvcLh8/ffnt7MPF/55dXXz6GF5enV1dIpUzGeMlTejwngzqTzxbA/WRZ/wpP/Ms/IF2Wc25gfreszWov/iMv+U3n6HFQ+0xC1f5Ukb2WECLpfapUGBzsJfpDF3WsMFup9AqsSdsjwIY3Nb1swIUHmQ1IrWwz+7LfhobLdCu/pGjwdSyTc54RP2pplD/Qi/Di6n8ny/e/g7/ciClvoMDsBtmabyFf3uiz9mv8gj0xjdz5VU4dZO3QHvINif6UZMq/aJ9dtWi7642Zxc0noCIMgqAiThXR/cBa0Cvcj6sdXrKP8rP8TpIXUEHAU6Y7qZ4UDrvroBkNpUdyE8oq68fwrb+TLtOcBpH/HX7nYbInxYFeWDVWD15T1yoWI7kzXJb6SwmoGyTjrUskiUB4U9rcDULs6u4AL8ZVXeVSBbzCFDBJfThCMg9PIBKG0IQhZ+aN1Cy+i3ujyCVxJkX+cYMJaWmCk8YJNK8ma/W6UwEQt3WO9O5qU0Jqjf5Z6Nmg8EHjcUwajaXgPzk+gujtrY2eApbP/XMzlgzbY5GidG+Yw1RzW2/Ndr1rDC07SlpKkZgLnXoRIJ5oC3VpAy3pVaha0PtnHR+Z3zdTipWNT2lqzyeQMWpaQ0Jn3R9l7I97TCHqFbDKarh5jG76DdhBwqlSq0IqqJjqdMq17IxQBgbXaQnSIsuGXfIe9DcAl6bIBs3FzE1kO0sHzYREwZKYvbley1/vvwcL1+EMbHFM6iiZpw0QkZIEvetTSaDH0RBy/evRihdyWitd1dqjFxWx4cG/sQagdlZKJNHX7dJjflGDjGqoyhWO25x8rrusOJrMLQJuE+o1piC2UAf+cQmDALG1T0OuyojPKi6yrNlo6bbtoq7Nn/KzmDjkE1j9w1sdkCgw9oase3Mugac9nneo7uI2z9xM7U/2Ga9gpXVx9CBX6tZ3kBpBx77mGE/LlXFBjZ74DyKUQPaD+BU9mzVSv1z0Nl1AKUP6VES7mqSJUuZJfHstuWWF3CW3yXPa5n/aEOyC+gLis9r17dfjT0rD1+EFUWtiCk3wI277g5Ry2R2rcNSI+ig7z4lo5++tU5bFwTuAtl9b2DXmAx1qpvmpQOERqOZAtRI7romJNps5scHGt2L1j099u0w7sWS4+Xdhf18fSE7+vpi2thkRIOo2RIG+WPepnLoWWA3dd3YWYMhH3UNB2MznDYcYHp7wyYoJAhdhagJ/NFB7R4RfeEQ1do062iL6h+jaWLz2Jt3R2IqBgIwME/YmoypaKoGGJXiq7AKpNjZsuZjY60x+ATYcrjdwA18Gjh9AvgGi3nqBBp7wHplOYQAcYCCymANVLklSOFE3VnmIQlcP3KPdyOHQ0+0E2B8/QM0IZ+FixJdgwj1byRpTua89EXUx0KepXc7mHd+2INO/Oi+iv5B2RjV3U6tN47siUORVzGwVLqZs1AqKHlHqQKd9Ilf9DEz0YoV7lrpRpyJqB90ui6bR4OqP+1TMg1jOqvoMmE+A/+Oz+LgW+Rp0wEMnoKtlyAiRXV0KqNg4Yps32SV5mPWIo/w3Gi95HA8ru9R0rZDH5zSB8BBOmzJaW/R5zRkrxL5Ahpr3K79Vz40brwlx7M+JmWVaS0fP3e4gNXNyrTE7J86xI0mRbj6jkjq5830vemGASmpkz/JyqSFV5wLkiKc19+30V60ZIw0sZBAeBGL6PaRJQR2cKjbrRhfr5yoKkSS7duxOfpWYKe08cZqLGYVQaVQQ9KrUUHNbGzOtNWbEPRjI2bRRJDZTqMBaKs9aRUfDFMekYYnHEjzlMJBn6PCDQ+7kr5D1coPl3tYGaWYI6AFwK9MwOhWRVAGJMkBngUHmYUJSDEQA1KD6SzQbTX30ev4Hv0fTj0RVyVXdVSTo5RZXdRXG5gcguq24Txf2FfYmzgWlyNyx96D9U/chaKARyBeXwFI2YA7xQJb+FXp5BK/xUyw9A0i/RN0xE9ZJZm2aKYLA2voPoyp/FxP20doHuk47f4lJvM4mtGlWBYHtbQ9H7d5MR+hm6oTW6IrBrRd4Udjre81VIUH3dhXevNsS0eZUmvg7z/0CFV00LS+A2F1fAiipy/p42sfKzk94+KPzjU2/YQXQfja+hbuldow8FPR27QbgZ9NXwAaOClFV+B6dM9D0pfwaVJkljo/C/SxpDIvmfK6eWMS42BJpjaNVpE2sWIF4hxF2zVKka3B/lSDRREniigRtekyN0dqvkLNsG47NTujYdC1aUzOzwOur4aI+2YLR2sjVlj70Jy+yCbE3nAkkSdWk4qQ9N3LWc/TGNDuWHFpn+kRPLKl+9DyMpVdoTs14pshPB3eMc94KROnxNm9p0736/USDdQRL22+NQXnF7JOHJVxNE86Kj2qRjd72amH74xGfgSM9jkq4WHp4FsdxlE/yGcys+5v2zcGI/IyKZqV2JakuEatR4OXPzUclMwzKQ4RM5Yj/EC9JT69tdngxUR2GznGeUz/qbzT6aM1eO+/heW2R6fRbqNftg9kqY007bbi9MRs/o3sYhfL6EDrtCvkT47qcfuhk2+I1sA3epzUwD7snRMAy8LuYDACcM1gZG1DEDZJrykFO3AgdK++IS3sdXQnplIG37tXn1b76wt04YianLCyzSjv6usLV+OIvcEh7YIncMZ2o2ezvZ4B/Qjr2wlK98eI0iczwF2A/y1M0BhSLxs06j3KCH8xTxWeyQz7D5l3hho5PWctvRrpv0sr7R6zRsjGUHYzyn6tE9XD/n6Ee0Z9W4rybx/1iNjv0miZ5XQVi2q7h23N1Ys5u26WWNsyYWeJumPP7hx/e6wiVZlyqmw0YyP6qEr4bYTGOJSmixTqf3ttm6lu5CLjY66sj0D6wjjlGYo65FKU3Xd40pOdpAZs/X0kA9TpG79zDpVXS65tGvHxoK7Q+K7jGCTyjoA5h+p2J1s8im0yRzTc9ky4cTAF49hxWtQW6/UB4r9VsPeL9h4a6BbvanQ/KOBVexDxrTNTKW9/UMYr2I9K+V/6ehaSvgclT5L2ahTB90dYDMt8VX+H1O+3PdU9/NLngzQ62GWx2Vf8MTw0r9MSNKtNdd/85HqoR4V1B1iK0YhYmUZUZMs2b0R4aYZ54wvOMoE2MNySzVia+YIOQ6RDTJwKLOSZwKLj6kHEDaIFO54vJg3oU9Blsshxfbxj1+097teRJ4nacHjDqP7r8tPHU2uec6bmdgaCoNpW/AF3+tSgQrIR8qZ/Ua8DNSoLuh9tbh+21TmBWpNmPHXn8TMims62PIHviaPcvMoV6VkOOWbkEtALcrbWi2G6guWpJd6CK/KKAHrjJgDO6FDZRvMFbZT+mePJj5mJa+z1v+vJrzF+BQZjwurEIxrVw/53QSgPRjos5xsGtVv4Sb6sxZM8WYt+ZeVpvixjjPKnf53UPkxqo3XUwLeRG8atn4ByTx9s5yw6GZVeJkDzlxQIPSWddnTu4D7u9mNc7lFUiO4m/VjAtCC6I5k+LUrzZBLplTr278Q4CIj4nDzH5Zxqnxv16uv0wKRDTYj4Fn0jpxA3N3M/mEGpJtQfxNrY/Du0YVGUVnHIto/11wai+3voTKtwlFuts5lp0vUHs2hrIjag/jlKy764equU5x0CXx4Q1A5ujjtlcd/XPXEjEvoiH3GX7MeRMOqC7y1sarK8GS3jyWepGYi4gpA+6MBZPrX97j13PZU5372qOy375kB/xKDvgqDb8cs+K76//Q+Y631x401rtlW/YXU2Gih7c2rECDUXzWhl1qhj3LpsgOWTLIDmEUurqvI66Ajo0+uX/5JWvxQ6fTNKq9PXLyq3lXTk9QaSVO77yJJaiV5KqS9DyoU/HI4eH/y5vD+AIFnacjmUdoJhPfSVSL5RoN1f62YBg898aUxfsJcm/p/KT753z3YASBYBaf+0+JJCxW/M3W7ezGPkCTqGIzLoOYLs8rJ4Ow20bkdn0P16N6h+GyroL9oNspHw01S1pZAJTBMjMCyNQPzd3ZM6vw4eyUIKuo/knoNlhtN1YvZM/DKgPkfz7vlKAzEwDEZPO+MP6p+7ofWlU/W83w3MSOB6RAoHplTeCdvcqYH5Qrugku1pJi+6/aJOiKSTbZnexjF1qF+1MuHIgVATqHyUmObnms44T/nJBKG+q2tYofTKRJvUQLTw0ycC71tELn02GdeA+3w3Mv+JrqnEVcFUq0ZKqs+FeN8kFNtqdo1vqfKaQds68dmZy2Ro/oQcEbkKYJNvPKu83i4Wq0TcbIo5eSF9GYie+2eXbdeh/FSPgKq9Aj6SZuIbb0FjdoG4CbN2yKAw4vE/xU0kU9OVEVXlubjChVzWoP3T7CwK3KGo0OaclHnSFdvEbcX9FMkmj6+FtBQD1GJLZFabyDJvxNi2k7p0nmNmdmrZyE9whMhUXDP+3jAbRbx904/BRqKrh8xS7C9/RE7PlaOoWPrd0bVyuPH3jgGH35IChPGXX36mLy28ePj/GnMgHA==')).decode('utf-8'))
SOURCE_HASHES = {'tile_data.py': '47059a7eeca1b7fd3edfa7bfe2a52bbab4ea6b506bcf322bb6962d163c1512be', 'test_tile_data.py': '40200ace3ea544567085f0edcbe218b53a69cb7caa86f10f1462a30d5b7b1a9e', 'run_benchmark.py': 'd686723ca406bf797593a69e8854657fd2b65f6f299affd48097217a88a4d0f5', 'captioning/scst_runtime.py': '3452ccfbd43876dc6381b78d1229d1d6b05fe723e3863a04f31c4a784497f669', 'captioning/visual_cache.py': 'adbcc9678fff499fc767cdbf04a4966186a40729f51c5bfb2525e79f9b6b5fe1', 'captioning/data.py': '9db178959dd9ed70312cfc2f58e11a0db28148d3328b2feea80786f8f2770463'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], name
    destination = REPO/name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
print('Pinned runtime and hash-checked embedded source READY', flush=True)


In [ ]:
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','h5py','Pillow','numpy','pandas'])
import torch
assert torch.cuda.device_count()==2,'Select GPU T4x2.'
assert all('T4' in torch.cuda.get_device_name(r) for r in range(2)),'Benchmark protocol requires two T4 GPUs.'
run([sys.executable,'-u','test_tile_data.py'],cwd=REPO)
CONFIG=Path('/tmp/native_detail_benchmark_config.json')
cfg=dict(checkpoint=str(CHECKPOINT),dataset_json=str(COCO_JSON),images=str(COCO_IMAGES),visual_cache=str(VISUAL_CACHE),
    output=str(OUTPUT),groups_archive=str(REPO/'experiments/noun_evidence_proposal/data/noun_evidence_train10k_v1_dataset.zip'),
    project_commit=PROJECT_COMMIT,implementation_sha256=SOURCE_HASHES,
    protocol='native_detail_cache_benchmark_v1',benchmark_only=True)
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
try:
    run([sys.executable,'-u','run_benchmark.py','run','--config',CONFIG],cwd=REPO)
finally:
    if OUTPUT.is_dir():
        shutil.copyfile(LOG,OUTPUT/'notebook.log')
        source_dir=OUTPUT/'source';source_dir.mkdir(exist_ok=True)
        for name,source in SOURCES.items():
            destination=source_dir/name;destination.parent.mkdir(parents=True,exist_ok=True)
            destination.write_text(source,encoding='utf-8')
        run([sys.executable,'-u','run_benchmark.py','archive','--config',CONFIG],cwd=REPO)
result=json.loads((OUTPUT/'benchmark_summary.json').read_text(encoding='utf-8'))
assert result['status']=='COMPLETE','Benchmark failed; return review ZIP with logs.'
print('BENCHMARK COMPLETE. No training or caption evaluation was run.',flush=True)
print('SEND:',str(OUTPUT)+'_review.zip',flush=True)
print('Full-cache estimate excludes training/evaluation; review quota before proceeding.',flush=True)
